# OctoSense Quickstart

Load and visualize one sequence from **[OctoSense](https://huggingface.co/datasets/anthonytec2/OctoSense)**, a time-synchronized multi-sensor dataset (stereo RGB, stereo event, IR, LiDAR, IMU, GPS, and CAN, with depth / segmentation / odometry ground truth).

This notebook downloads all eight files for a single sequence: `data.h5`, `img_left.mp4`, `img_right.mp4`, `img_infrared.mp4`, `events.h5`, `captions.h5`, `rgb_left_rect_depth.h5`, and `rgb_left_rect_semantic.h5`. Nighttime sequences have only seven files, as no semantic segmentation is available at night.

Aside from our sensor data our dataset has perception for depth estimation, semantic segmentation, optical flow estimation, ego-motion / odometry estimation, and image captioning.

In [ ]:
# 1) Install deps
!pip -q install huggingface_hub h5py hdf5plugin opencv-python-headless matplotlib numpy pyproj contextily
!pip -q install torch==2.12.0 torchvision==0.27.0 torchcodec==0.14.0 --index-url https://download.pytorch.org/whl/cpu


In [ ]:
# 2) Imports + config
import os, json, numpy as np, h5py, hdf5plugin, cv2, torch, pyproj # hdf5plugin needed to read our compressed h5 fields
import matplotlib.pyplot as plt
import contextily as cx # Used for our base GPS maps
from huggingface_hub import hf_hub_download
from torchcodec.decoders import VideoDecoder # Used to read our mp4 files, for GPU users download the GPU variant
import pandas as pd
REPO = "anthonytec2/OctoSense"
print("ready")

def grid_display(items, ncols=8):
    """Display a flat iterable as a compact, index-free grid.

    items : any iterable (list, dict_keys, np.array, etc.)
    ncols : number of columns in the grid
    """
    items = [str(x) for x in items]
    padded = items + [""] * (-len(items) % ncols)
    grid = np.array(padded).reshape(-1, ncols)
    return (pd.DataFrame(grid)
              .style.hide(axis=0).hide(axis=1))

## Browse and select a sequence

At the top level of the Hugging Face dataset we provide `metadata.jsonl`, a 271 kB file with one record per sequence (371 in total).

Some useful keys:
- `duration_s`: the span of the longest-running sensor in the sequence.
- `sensor_dropout`: flags any sensors that dropped out during recording.
- `distance_m`: total estimated driving distance from RKO-LIO odometry.
- `session`: the session a sequence belongs to (used to index and download it).
- `idle_fraction`: fraction of the sequence spent below 1 mph (many bags are city driving, which is heavily stop-and-go).
- `gps_lat_[min,max]`, `gps_lon_[min,max]`: a bounding box of the area driven.

In [ ]:
# 3) Get metadata, and showcase the keys for each sequence.
mj = hf_hub_download(REPO, "metadata.jsonl", repo_type="dataset")
recs = [json.loads(l) for l in open(mj)]
print(f"{len(recs)} sequences total\n")

grid_display(recs[10])

In [ ]:
# 4) Pick a bag.  Set BAG = "<session>/<bag_id>" from the list above
BAG = "car/sess8/rosbag2_2026_01_09-11_32_05"
bag_id = BAG.split("/")[-1]
rec = next(r for r in recs if r["bag_id"] == bag_id)
print("using bag:", BAG, "| speed=", rec["mean_speed_mph"],"mph ", "distance=", rec["distance_m"], "m ", "duration=", rec["duration_s"],"s ", "day=", rec['is_daytime'], "seg=", rec["has_seg"], )
print("gps coordinages (lat min, lat max, lon min, lon max): ",  rec["gps_lat_min"], rec["gps_lat_max"], rec["gps_lon_min"], rec["gps_lon_max"])

> **Download size:** one sequence ranges from a few GB up to ~25 GB, and `events.h5` is ~78% of that. If you don't need the raw events, drop `"events.h5"` from the `files` list below to download much faster.

In [ ]:
# 5) Download the files for your selected sequence (this can take a while for large sequences)
files = ["data.h5", "img_left.mp4", "img_right.mp4", "img_infrared.mp4", "events.h5",  "rgb_left_rect_depth.h5", "captions.h5"]
if rec.get("has_seg"):
    files.append("rgb_left_rect_semantic.h5")
paths = {}
for f in files:
    print("downloading", f, "...")
    paths[f] = hf_hub_download(REPO, f"{BAG}/{f}", repo_type="dataset")
print("\nsizes (MB):", {f: round(os.path.getsize(p)/1e6, 1) for f, p in paths.items()})

## data.h5

`data.h5` holds the sensor data: IMU, LiDAR, GPS, CAN, RGB and infrared metadata, the fused GPS + RKO-LIO trajectory, and the sensor intrinsics and extrinsics.

It has eight top-level groups:
  - `calib`: sensor extrinsics (and per-camera intrinsics).
  - `car`: CAN-bus data tapped from the vehicle.
  - `fused_traj`: the fused GPS + RKO-LIO trajectory.
  - `gps`: GPS fixes and ENU velocity.
  - `img`: left and right RGB intrinsics, extrinsics, and timestamps (pixels are in `img_left.mp4` / `img_right.mp4`).
  - `infrared`: infrared intrinsics, extrinsics, and timestamps (pixels in `img_infrared.mp4`).
  - `ouster`: all of the LiDAR data, plus the in-LiDAR IMU and RKO-LIO odometry.
  - `vectornav`: the VN-100 IMU data.

The event-camera data lives in `events.h5`. A seperate file to allow for selective download.

In [ ]:
# 6) Peek at data.h5 structure
f = h5py.File(paths["data.h5"], "r")
print("top-level groups:", sorted(f.keys()))
print("\nkey arrays this notebook reads:")
for k in ["img/left/t", "img/left/intrinsics", "img/left/dist_coeffs",
          "infrared/t", "infrared/intrinsics",
          "ouster/t", "ouster/range_pcl", "ouster/sig_pcl",
          "ouster/imgl_T_ouster", "ouster/odom/map_T_lidart",
          "vectornav/t", "vectornav/accel", "vectornav/ang_vel",
          "gps/data", "gps/velocity_enu",
          "car/speed", "calib/imgl_T_imgr", "fused_traj/T_world_lidar"]:
    if k in f:
        print(f"  {k:30s} {f[k].shape}  {f[k].dtype}")


## Working with the data

**Aligning sensors by time.** Every stream is timestamped on one main clock (`*/t` arrays in seconds; events in microseconds). Frame `j` of `img_[left,right, infrared].mp4` corresponds to `f["img/[left,right, infrared]/t"][j]`. To grab any sensor at an instant `t`, take the nearest timestamp:
```python
def get_nearest(t, ts): # t query time, ts timestamp array of other sensor
  return int(np.argmin(np.abs(ts - t)))
```

**Units.** depth cm (`0` = invalid), event time microseconds, all other `*/t` seconds, IMU accel $m/s^2$, gyro rad/s, magnetometer Gauss, pressure Pa, temperature °C, LiDAR XYZ mm, odometry translation m, CAN speed mph, steering deg.

**Gotchas.** Import `hdf5plugin` before reading the Blosc-compressed h5 files; `.mp4` decoding uses `torchcodec`. A few sequences have no GPS or CAN; some event cameras have dropped data, all these sensor issues should be viewable from `sensor_dropout` metadata column.

### `Data.h5` - `calib` group

<div align="center">
  <img src="https://huggingface.co/datasets/anthonytec2/OctoSense/resolve/main/assets/cal_axis.png"
       alt="Sensor frames and calibration axes" width="500">
</div>

All extrinsics live per-sequence under `calib`, each named `A_T_B`: a 4×4 matrix
that maps a point from frame **B** into frame **A**.  We showcase a frame diagram of each of the sensors above with the RGB axis coloring corresponding to the XYZ axes. For an introduction to rigid body transformation checkout the online course material of [MEAM620 Lectures 1-4](https://alliance.seas.upenn.edu/~meam620/wiki/index.php?n=Main.Schedule).

> **Naming convention:** `A_T_B` transforms **B → A**.
> e.g. `imgl_T_ouster` takes LiDAR points into the left-RGB frame.

**Frame abbreviations**
| abbrev | frame | abbrev | frame |
|---|---|---|---|
| `imgl` / `imgr` | RGB left / right | `ir` | Infrared (FLIR A35) |
| `evl` / `evr` | Event left / right | `ouster` | LiDAR |
| `imu` | VectorNav IMU | | |

In [ ]:
print("evl_T_evr: \n", f['calib']['evl_T_evr'][:], "\n")
print("Calibration Fields: \n")
grid_display(f['calib'],4)

### `data.h5` - `car` group

Cars exchange data between sensors over a CAN bus. For this dataset we tapped the forward-sensing-camera CAN line of a 2021 Mazda CX-5 and recorded all traffic on it. Each [CAN message](https://github.com/autowarefoundation/ros2_socketcan/blob/main/ros2_socketcan_msgs/msg/FdFrame.msg) carries a numeric ID identifying the source plus a data payload. To decode the payloads we use a DBC file, which maps the numeric IDs to named signals, here [opendbc](https://github.com/commaai/opendbc)'s `mazda_2017.dbc` and `mazda_radar.dbc`. In our experience these decode our 2021 vehicle correctly, though many records are in arbitrary units.

In [ ]:
print("Decoded Data Records for our car")
grid_display(f['car'],4)

In [ ]:
import matplotlib.pyplot as plt

# Define signals to plot (excluding radar as requested)
car_signals = [k for k in f['car'].keys() if k != 'radar']

# Map signals to requested units using LaTeX formatting
units = {
    'brake_on': 'AU', # Brake Pedal Active
    'brake_press': 'AU', # Intensity on the break pedal
    'pedal': 'AU', # Acceleration pedal intensity
    'speed': 'mph',
    'steer': r'$^\circ$', # Steering Angle of the wheel, can get greater than 360 for mutiple wheel turns
    'steer_rate': r'$^\circ/s$', # Rate at which steering angle is changing
    'vcc': r'$m/s^2$', # Body Acceleration (acc_x, acc_y)
    'wheels': 'mph' # Speed of each of the wheels (FL, FR, RL, RR)
}

fig, axs = plt.subplots(3, 3, figsize=(18, 12))
axs = axs.flatten()

for i, signal in enumerate(car_signals):
    data = f['car'][signal][:]
    t = data[:, 0]

    # Plotting logic for multi-field data
    if signal == 'wheels' and data.shape[1] > 2:
        labels = ['FL', 'FR', 'RL', 'RR']
        for j in range(1, data.shape[1]):
            lbl = labels[j-1] if j-1 < len(labels) else f'W{j}'
            axs[i].plot(t, data[:, j], alpha=0.7, label=lbl)
        axs[i].legend(loc='upper right', fontsize='small')
        axs[i].set_title("Wheel Speed")
    elif signal == 'vcc' and data.shape[1] > 2:
        axs[i].plot(t, data[:, 1], label=r'$acc_x$')
        axs[i].plot(t, data[:, 2], label=r'$acc_y$')
        axs[i].legend(loc='upper right', fontsize='small')
        axs[i].set_title('Vehicle Acceleration')
    else:
        axs[i].plot(t, data[:, 1])
        axs[i].set_title(signal)

    axs[i].set_xlabel('Time (s)')
    axs[i].set_ylabel(units.get(signal, 'Value'))
    axs[i].grid(True)

# Hide unused subplots
for j in range(len(car_signals), len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()

The CAN data also gives us processed radar point tracks (decoded with `mazda_radar.dbc`). These appear to correlate with other vehicles in the scene, but we cannot verify their units. Treat this field as experimental.

In [ ]:
SENTINEL = 4095 # This indicates the point track reading is invalid in distance units
DIST_M_PER_COUNT, HALF_FOV_DEG = 1.0, 30.0     # raw counts + assumed FOV (units unverified)
T_SEL = 50 # timestamp to plot

t    = f["car/radar/track_1/t"][:]  # shared radar clock
dist = np.stack([f[f"car/radar/track_{i}/dist"][:].astype(float) for i in range(1,7)])  # (6, N) Distance of the object
ang  = np.stack([f[f"car/radar/track_{i}/ang"][:].astype(float)  for i in range(1,7)]) # Angle of the object
vrel = np.stack([f[f"car/radar/track_{i}/vrel"][:].astype(float) for i in range(1,7)]) # Relative velocity of the object

k = int(np.argmin(np.abs(t - T_SEL)))
m = dist[:, k] != SENTINEL # valid tracks at this frame
r, th = dist[m, k] * DIST_M_PER_COUNT, np.radians(ang[m, k] / 2047.0 * HALF_FOV_DEG)
x, y, v = r * np.sin(th), r * np.cos(th), vrel[m, k]

fig, ax = plt.subplots(figsize=(7, 7)); Rmax = (y.max() if len(y) else 1) * 1.15

# Plot Cone
for s in (-1, 1):
    a = np.radians(s * HALF_FOV_DEG); ax.plot([0, Rmax*np.sin(a)], [0, Rmax*np.cos(a)], "k--", alpha=.3)
ax.plot(0, 0, "^k", ms=16)


# Plot point tracks
if len(x):
    vlim = max(abs(v).max(), 1)
    sc = ax.scatter(x, y, c=v, cmap="coolwarm", vmin=-vlim, vmax=vlim, s=160, edgecolor="k", zorder=3)
    fig.colorbar(sc, label="$RELV_{OBJ}$ (raw)")
ax.set_aspect("equal"); ax.grid(alpha=.3)
ax.set_xlabel("lateral = dist·sinθ"); ax.set_ylabel("forward = dist·cosθ")
ax.set_title(f"Radar @ t={t[k]:.1f}s  ({len(x)} objects)"); plt.tight_layout();

### `data.h5` - `fused_traj` group

We have several sources of ego-motion: GPS, RKO-LIO, and IMU. In `fused_traj` we fuse them with a `gtsam` factor graph to produce a refined trajectory grounded in GPS (UTM) coordinates. Depending on GPS quality, some residual drift relative to the GPS map can remain.

A few sequences have degraded GPS and therefore contain RKO-LIO odometry only.

In [ ]:
g = f["fused_traj"]
T  = g["T_world_lidar"][:]
epsg = int(g.attrs["epsg"])  # UTM zone (e.g. 32618 = UTM 18) origin of the data
east  = T[:, 0, 3] + float(g.attrs["origin_easting_m"])
north = T[:, 1, 3] + float(g.attrs["origin_northing_m"])

fig, ax = plt.subplots(figsize=(9, 9))

ax.scatter(east, north, c=np.arange(len(east)), cmap="viridis", s=6, zorder=3)

ax.plot(east[0], north[0], "o", color="lime", ms=12, mec="k", zorder=4, label="start")
ax.plot(east[-1], north[-1], "s", color="k", ms=10, zorder=4, label="end")
ax.set_aspect("equal")
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery,crs=f"EPSG:{epsg}", zoom=17)

ax.set_axis_off(); ax.legend(loc="upper right"); ax.set_title("fused_traj on satellite basemap")

### `data.h5` - `gps` group

We use a u-blox ZED-F9P receiver fed with NTRIP corrections for RTK-grade fixes. GPS and NTRIP corrections are not available at all times during a sequence. The group provides latitude, longitude, altitude, and velocity in East-North-Up (ENU) coordinates.

In [ ]:
g = f["gps"]
data = g["data"][:]
t = g["t"][:]
vel, vt = g["velocity_enu"][:], g["velocity_t"][:]
m = data[:, 6] >= 0

lon, lat = data[m, 1], data[m, 0]


x, y = pyproj.Transformer.from_crs(4326, 32618, always_xy=True).transform(lon, lat) # GPS ref to UTM 18
vE = np.interp(t[m], vt, vel[:, 0]); vN = np.interp(t[m], vt, vel[:, 1])
spd = np.hypot(vE, vN) * 2.237 # convert to mph

fig, ax = plt.subplots(figsize=(10, 10))
sc = ax.scatter(x, y, c=spd, marker="o", cmap="turbo", s=12, zorder=3) # plot the GPS fixes
fig.colorbar(sc, label="speed (mph)")
s = max(len(x)//40, 1)
ax.quiver(x[::s], y[::s], vE[::s], vN[::s], color="white", zorder=4,  # Plot the velocity vectors every 40th pulse
          scale=350, width=0.004, edgecolor="k", linewidth=0.4)
ax.plot(x[0], y[0], "o", color="lime", ms=13, mec="k", zorder=5, label="start")
ax.set_aspect("equal"); ax.set_axis_off(); ax.legend(loc="upper right")
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, crs=f"EPSG:{epsg}", zoom=17)
ax.set_title("GPS track,  color = speed, arrows = ENU velocity")

### `data.h5` - `img` group

The `img` group has two subgroups, `left` and `right`. Each carries its own `dist_coeffs`, `intrinsics`, and `t` timestamps (all sensors share one main-clock timeline). The pixel data lives in `img_left.mp4` / `img_right.mp4`, where each frame index maps directly to one entry in the `t` array.

In [ ]:
# Read
idx = 100

vl=VideoDecoder(paths["img_left.mp4"])
vr=VideoDecoder(paths["img_right.mp4"])

imgl = vl[idx].permute(1, 2, 0).cpu().numpy()   # CHW tensor -> HWC numpy
imgr = vr[idx].permute(1, 2, 0).cpu().numpy()


plt.figure(figsize=(12,8))
plt.subplot(1,2,1)
plt.imshow(imgl)
plt.title(f"Left Img, t={f['img']['left']['t'][idx]:.03f}")
plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(imgr)
plt.title(f"Right Img, t={f['img']['right']['t'][idx]:.03f}")
plt.axis('off')

In [ ]:
D_img_left = f["img/left/dist_coeffs"][:]
intr_img_left = f["img/left/intrinsics"][:]
res = f["img/left/resolution"][:]

D_img_right = f["img/right/dist_coeffs"][:]
intr_img_right = f["img/right/intrinsics"][:]

imgr_T_imgl = np.linalg.inv(f['calib']['imgl_T_imgr'][:])
(rectl_R_rawl, rectr_R_rawr,
 P_rect_left, P_rect_right) = cv2.stereoRectify(
    cameraMatrix1=intr_img_left,  cameraMatrix2=intr_img_right,
    distCoeffs1=D_img_left,       distCoeffs2=D_img_right,
    imageSize=res,
    R=imgr_T_imgl[:3, :3], T=imgr_T_imgl[:3, -1],
    flags=cv2.CALIB_ZERO_DISPARITY, alpha=0)[:4]


mapx_left, mapy_left = cv2.initUndistortRectifyMap(
    intr_img_left,  D_img_left,  rectl_R_rawl, P_rect_left,  res, cv2.CV_32FC1)
mapx_right, mapy_right = cv2.initUndistortRectifyMap(
    intr_img_right, D_img_right, rectr_R_rawr, P_rect_right, res, cv2.CV_32FC1)


# --- rectify a frame ---
rectl = cv2.remap(imgl, mapx_left, mapy_left, cv2.INTER_LINEAR)
rectr = cv2.remap(imgr, mapx_right, mapy_right, cv2.INTER_LINEAR)

plt.figure(figsize=(12, 8))
plt.subplot(1, 2, 1)
plt.imshow(rectl)
plt.title(f"Left Img rectified, t={f['img']['left']['t'][idx]:.03f}")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(rectr)
plt.title(f"Right Img rectified, t={f['img']['right']['t'][idx]:.03f}")
plt.axis('off')

### `data.h5` - `infrared` group

The `infrared` camera has its own `dist_coeffs`, `intrinsics`, and `t` timestamps (on the same main-clock timeline as every other sensor). The pixel data lives in `img_infrared.mp4`, where each frame index maps directly to one entry in the `t` array.

In [ ]:
# Read
idx = 100

vi=VideoDecoder(paths["img_infrared.mp4"])

imgi = vi[idx].permute(1, 2, 0).cpu().numpy()   # CHW tensor -> HWC numpy

plt.imshow(imgi)
plt.title(f"Infrared Img, t={f['infrared']['t'][idx]:.03f}")
plt.axis('off')

In [ ]:
D_infrared = f["infrared/dist_coeffs"][:]
intr_infrared = f["infrared/intrinsics"][:]

undist_infrared = cv2.undistort(imgi, intr_infrared, D_infrared)
plt.imshow(undist_infrared)
plt.title(f"Undistorted Infrared Img, t={f['infrared']['t'][idx]:.03f}")
plt.axis('off')

### `data.h5` - `ouster` group

The `ouster` group holds the data from our Ouster OS1-64 LiDAR. Each scan is 64 beams × 2048 azimuth columns = 131,072 points, captured at ≈10 Hz and destaggered (columns realigned into a rectangular grid):
- `range_pcl` (N, 131072, 3) int32 - the Cartesian XYZ of each return, in millimeters (`[0, 0, 0]` = no return).
- `sig_pcl` (N, 131072) uint16 - signal intensity per point (strength of the laser return).
- `nir_pcl` (N, 131072) uint16 - near-infrared per point (ambient-light intensity).
- `refl_pcl` (N, 131072) uint8 - reflectivity per point.
- `t` (N,) - per-scan timestamp (main-clock seconds).

The OS1-64 also carries its own IMU (IAM-20680HT), logged as `accel` (m/s²) and `ang_vel` (rad/s) at ≈100 Hz, each with its own timestamps (`accel_t`, `ang_t`).

Finally, `ouster/odom` holds the RKO-LIO odometry: the LiDAR pose in the map frame as `map_T_lidart` (N, 4, 4) (translation in meters), plus body-frame `lin_vel` (m/s) and `ang_vel` (rad/s) at the per-scan rate. `ouster/hf_odom` is the same quantities at higher rate, integrating the IMU between LiDAR scans.

In [ ]:
print("Decoded Data Records for our car")
grid_display(f['ouster'],4)

In [ ]:
# Project the LiDAR point cloud into the rectified left image, colored by depth.
idx = 100

t = f['ouster/t'][idx]
range_raw = f['ouster/range_pcl'][idx]

# 1) Scale to meters and filter zero-returns
range_m = range_raw.astype(np.float64) / 1000.0
valid_mask = np.linalg.norm(range_m, axis=1) > 0
range_m = range_m[valid_mask]

# Getting Base Image to overlay LIDAR data
img_idx = np.searchsorted(f['img/left/t'][:], t)
img_lidar = vl[img_idx]

imgl_T_ouster = f['ouster/imgl_T_ouster'][:]

# 2) Transform points from lidar to camera frame
# range_m is (N, 3). Add homogeneous 1s: (N, 4)
points_lidar_hom = np.concatenate([range_m, np.ones((range_m.shape[0], 1))], axis=1)
points_cam = (imgl_T_ouster @ points_lidar_hom.T).T

# 3) Transform points into the rectified frame using rectl_R_rawl and project
# Project using the Rectified Projection Matrix P_rect_left
pts_rect = (P_rect_left[:, :3] @ rectl_R_rawl @ points_cam[:, :3].T).T

# Filter points in front of camera
valid_depth = pts_rect[:, 2] > 0.5
pts_rect = pts_rect[valid_depth]

# Normalize by depth to get (u, v)
u = pts_rect[:, 0] / pts_rect[:, 2]
v = pts_rect[:, 1] / pts_rect[:, 2]
depth = pts_rect[:, 2]

# 4) Filter by image boundaries
valid_uv = (u >= 0) & (v >= 0) & (u < res[0]) & (v < res[1])
u, v, depth = u[valid_uv], v[valid_uv], depth[valid_uv]

# Rectify the base image
rectl = cv2.remap(img_lidar.permute(1, 2, 0).numpy(), mapx_left, mapy_left, cv2.INTER_LINEAR)

plt.figure(figsize=(15, 10))
plt.imshow(rectl)
plt.scatter(u, v, c=depth, cmap='turbo', s=1, alpha=0.5)
plt.title(f"LiDAR Projection on Rectified Left Img (t={t:.2f}s)")
plt.axis('off')

In [ ]:
# Project LiDAR into the rectified left image, colored by each per-point channel (range / signal / reflectivity / NIR).
idx = 100
t = f['ouster/t'][idx]

# 1) Get raw data and masks
range_raw = f['ouster/range_pcl'][idx]
sig_raw = f['ouster/sig_pcl'][idx]
nir_raw = f['ouster/nir_pcl'][idx]
refl_raw = f['ouster/refl_pcl'][idx]

range_m = range_raw.astype(np.float32) / 1000.0
dists = np.linalg.norm(range_m, axis=1)
valid_mask = dists > 0

# 2) Project all points into Rectified Image Space (UV)
pts_lidar_hom = np.concatenate([range_m[valid_mask], np.ones((np.sum(valid_mask), 1))], axis=1)
pts_cam = (imgl_T_ouster @ pts_lidar_hom.T).T
pts_rect = (P_rect_left[:, :3] @ rectl_R_rawl @ pts_cam[:, :3].T).T

# 3) Filter depth and boundaries
valid_depth = pts_rect[:, 2] > 0.5
u = pts_rect[valid_depth, 0] / pts_rect[valid_depth, 2]
v = pts_rect[valid_depth, 1] / pts_rect[valid_depth, 2]
depth = pts_rect[valid_depth, 2]

valid_uv = (u >= 0) & (v >= 0) & (u < res[0]) & (v < res[1])
u_final, v_final = u[valid_uv], v[valid_uv]

# 4) Extract corresponding attributes for the valid pixels
sig_final = sig_raw[valid_mask][valid_depth][valid_uv]
nir_final = nir_raw[valid_mask][valid_depth][valid_uv]
refl_final = refl_raw[valid_mask][valid_depth][valid_uv]
range_final = depth[valid_uv] # Using projected Z as range for visualization

# Rectify image
img_idx = np.searchsorted(f['img/left/t'][:], t)
img_lidar = vl[img_idx]
rect_img = cv2.remap(img_lidar.permute(1, 2, 0).numpy(), mapx_left, mapy_left, cv2.INTER_LINEAR)

# Setup plot
fig, axs = plt.subplots(2, 2, figsize=(24, 16))

modalities = [
    ('Range', range_final, 'turbo', (0, 40)),
    ('Signal', sig_final, 'magma', (0, np.percentile(sig_final, 95))),
    ('Reflectivity', refl_final, 'viridis', (0, np.percentile(refl_final, 95))),
    ('NIR', nir_final, 'gray', (0, np.percentile(nir_final, 95)))
]

for i, (name, data, cmap, clim) in enumerate(modalities):
    ax = axs[i//2, i%2]
    ax.imshow(rect_img)
    sc = ax.scatter(u_final, v_final, c=data, cmap=cmap, s=0.5, alpha=0.6, vmin=clim[0], vmax=clim[1])
    ax.set_title(f"Projected {name}")
    ax.axis('off')
    plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)

plt.suptitle(f"LiDAR Projections by Modality @ t={t:.2f}s", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
odom = f['ouster']['odom']['map_T_lidart'][:]
t = f['ouster']['odom']['t'][:]

# Extract X and Y positions (translation components from the 4x4 matrix)
x_pos = odom[:, 0, 3]
y_pos = odom[:, 1, 3]

fig, ax = plt.subplots(figsize=(10, 8))

# Use a scatter plot with color mapped to time to show progression
sc = ax.scatter(x_pos, y_pos, c=t, cmap='viridis', s=10, label='Path')
cbar = fig.colorbar(sc)
cbar.set_label('Time (s)')

# Mark the start and end of the trajectory
ax.plot(x_pos[0], y_pos[0], 'go', ms=10, label='Start', mec='k')
ax.plot(x_pos[-1], y_pos[-1], 'ks', ms=8, label='End')

ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_xlabel("X Position (m)")
ax.set_ylabel("Y Position (m)")
ax.set_title("RKO-LIO Odometry Trajectory")
ax.legend()


### `data.h5` - `vectornav` group

The `vectornav` group holds the data from our VectorNav VN-100 IMU: a 3-axis accelerometer (m/s²), gyroscope (rad/s), magnetometer (Gauss), barometer (Pa), and temperature sensor (°C), logged at 400 Hz. The VN-100's onboard filter can bias-compensate both the accelerometer and gyroscope, so we store both the compensated arrays (`accel`/`ang_vel`) and the uncompensated raw arrays (`accel_raw`/`ang_vel_raw`).

In [ ]:
t = f['vectornav']['t'][:]
fig, axs = plt.subplots(3, 2, figsize=(15, 12))
axs = axs.flatten()

# 1) Accelerometer
axs[0].plot(t, f['vectornav']['accel'][:, 0], label=r'$a_x$')
axs[0].plot(t, f['vectornav']['accel'][:, 1], label=r'$a_y$')
axs[0].plot(t, f['vectornav']['accel'][:, 2], label=r'$a_z$')
axs[0].set_title('Acceleration')
axs[0].set_ylabel(r'$m/s^2$')
axs[0].legend(loc='upper right', fontsize='small')

# 2) Gyroscope
axs[1].plot(t, f['vectornav']['ang_vel'][:, 0], label=r'$\omega_x$')
axs[1].plot(t, f['vectornav']['ang_vel'][:, 1], label=r'$\omega_y$')
axs[1].plot(t, f['vectornav']['ang_vel'][:, 2], label=r'$\omega_z$')
axs[1].set_title('Angular Velocity')
axs[1].set_ylabel(r'$rad/s$')
axs[1].legend(loc='upper right', fontsize='small')

# 3) Magnetometer
axs[2].plot(t, f['vectornav']['magnetic'][:, 0], label=r'$m_x$')
axs[2].plot(t, f['vectornav']['magnetic'][:, 1], label=r'$m_y$')
axs[2].plot(t, f['vectornav']['magnetic'][:, 2], label=r'$m_z$')
axs[2].set_title('Magnetic Field')
axs[2].set_ylabel('Gauss')
axs[2].legend(loc='upper right', fontsize='small')

# 4) Pressure
axs[3].plot(t, f['vectornav']['pressure'][:])
axs[3].set_title('Barometric Pressure')
axs[3].set_ylabel('Pa')

# 5) Temperature
axs[4].plot(t, f['vectornav']['temperature'][:])
axs[4].set_title('Internal Temperature')
axs[4].set_ylabel(r'$^\circ C$')

# Formatting
for i in range(5):
    axs[i].set_xlabel('Time (s)')
    axs[i].grid(True, alpha=0.3)

# Hide empty subplot
axs[5].axis('off')

plt.tight_layout()
plt.show()

### `events.h5`

To make it easy to download sensor subsets, the event data lives in its own file. Like the RGB cameras, it has two top-level groups, `left` and `right`, one per event camera. Each contains:
- `(x, y)` - the pixel coordinates of each event (native 640×480).
- `p` - the event polarity (0 = OFF, 1 = ON).
- `t` - the event time in microseconds (main clock).
- `ms_to_idx` - a lookup mapping each millisecond to its start index in the event arrays.
- `intrinsics`, `dist_coeffs`, `resolution` - the event-camera calibration data.

In [ ]:
f_ev = h5py.File(paths["events.h5"], "r")

ms_time_low = 5000
ms_time_high = 5040

# Get indices for the time window
idx_low = f_ev["ev/left/ms_to_idx"][ms_time_low]
idx_high = f_ev["ev/left/ms_to_idx"][ms_time_high]

# Extract data
x = f_ev["ev/left/x"][idx_low:idx_high]
y = f_ev["ev/left/y"][idx_low:idx_high]
p = f_ev["ev/left/p"][idx_low:idx_high]

# Create a high-quality visualization
plt.figure(figsize=(14, 9), facecolor='#111111')

# Split polarities for distinct coloring
pos = p == 1
neg = p == 0

# Plot positive events in light blue and negative in coral/red
plt.scatter(x[pos], y[pos], color='deepskyblue', s=0.3, alpha=0.7, label='Positive (ON)')
plt.scatter(x[neg], y[neg], color='tomato', s=0.3, alpha=0.7, label='Negative (OFF)')

plt.gca().set_facecolor('#111111')
plt.title(f"Event Stream (Left Camera) | {ms_time_low}-{ms_time_high}ms", color='white', fontsize=16, pad=20)
plt.xlim(0, 640)
plt.ylim(480, 0)
plt.axis('off')

lgnd = plt.legend(loc="upper right", scatterpoints=1, markerscale=40, facecolor='#222222', edgecolor='white', labelcolor='white')


In [ ]:
# Event-camera calibration now lives in events.h5 (self-contained), not data.h5
D_ev_l = f_ev['ev/left/dist_coeffs'][:]
K_ev_l = f_ev['ev/left/intrinsics'][:]
res_ev = f_ev['ev/left/resolution'][:]

D_ev_r = f_ev['ev/right/dist_coeffs'][:]
K_ev_r = f_ev['ev/right/intrinsics'][:]

ev_r_T_evl = np.linalg.inv(f['calib']['evl_T_evr'][:])

R1, _, P1, _, _, _, _ = cv2.stereoRectify(
    K_ev_l, D_ev_l, K_ev_r, D_ev_r, res_ev,
    ev_r_T_evl[:3, :3], ev_r_T_evl[:3, 3], alpha=0
)

# Rectify the sparse event points with cv2.undistortPoints
pts_raw = np.stack([x, y], axis=-1).reshape(-1, 1, 2).astype(np.float32)
pts_rect = cv2.undistortPoints(pts_raw, K_ev_l, D_ev_l, R=R1, P=P1).reshape(-1, 2)
x_rect, y_rect = pts_rect[:, 0], pts_rect[:, 1]

plt.figure(figsize=(12, 8), facecolor='#111111')
plt.scatter(x_rect[p == 1], y_rect[p == 1], c='deepskyblue', s=0.5, alpha=0.8, label='ON')
plt.scatter(x_rect[p == 0], y_rect[p == 0], c='tomato', s=0.5, alpha=0.8, label='OFF')
plt.gca().set_facecolor('#111111')
plt.title("Rectified event stream (left)", color='white', fontsize=14)
plt.xlim(0, res_ev[0]); plt.ylim(res_ev[1], 0); plt.axis('off')
plt.legend(loc="upper right", scatterpoints=1, markerscale=40, facecolor='#222222', edgecolor='white', labelcolor='white')

### `captions.h5`

Each sequence is split into short windows; for every window we provide a scene caption (Gemma VLM) and a 4096-d text embedding (Qwen3-Embedding-8B) of that caption, which enables natural-language and semantic search over the dataset.
- `captions` (W,): caption text per window.
- `embeddings` (W, 4096): caption embedding per window.
- `metadata` (W,): `window_id`, `frame_idx` (into `img_left.mp4`), `timestamp` (s), `speed_mps`, `turn_deg`, `dist_m`, `is_night`.

To search by text, embed your query with the same Qwen3-Embedding-8B model and rank windows by cosine similarity against `embeddings`.

In [ ]:
fc   = h5py.File(paths["captions.h5"], "r")
caps = fc["captions"][:]; meta = fc["metadata"][:]; emb = fc["embeddings"][:]
print(f"{fc.attrs['num_windows']} windows | captions: {fc.attrs['caption_model']} | "
      f"embeddings: {fc.attrs['embed_model']} ({fc.attrs['embed_dim']}-d)")

wi = 0; fr = int(meta["frame_idx"][wi]) # RGB frame for one caption window
cap0 = caps[wi].decode()
plt.figure(figsize=(9, 6)); plt.imshow(vl[fr].permute(1, 2, 0).cpu().numpy()); plt.axis("off")
plt.title(cap0, fontsize=9, wrap=True); plt.show()

## Time sync

Every sensor is timestamped on a single PPS-synchronized **main clock**, so the streams can be aligned by time. Below we pick one wall-clock instant `t₀` and pull the nearest sample from each modality - stereo RGB, stereo event, infrared, LiDAR, and IMU, then show them together.

In [ ]:
# Pick one wall-clock instant t0; pull the nearest sample from every stream.
t0 = float(f["ouster/t"][len(f["ouster/t"]) // 2])          # mid-sequence instant (s, main clock)
near = lambda ts, t: int(np.argmin(np.abs(np.asarray(ts) - t)))

# stereo RGB + IR: nearest video frame to t0
il  = near(f["img/left/t"][:],  t0)
ir_ = near(f["img/right/t"][:], t0)
ii  = near(f["infrared/t"][:],  t0)
rgb_l = vl[il].permute(1, 2, 0).cpu().numpy()
rgb_r = vr[ir_].permute(1, 2, 0).cpu().numpy()
ir_im = vi[ii].permute(1, 2, 0).cpu().numpy()

# event image: accumulate +/-20 ms around t0 into a polarity image (per side)
def event_image(side, t0, dt=0.02):
    ms2idx = f_ev[f"ev/{side}/ms_to_idx"]; n = len(ms2idx)
    lo = int(ms2idx[int(np.clip((t0 - dt) * 1000, 0, n - 1))])
    hi = int(ms2idx[int(np.clip((t0 + dt) * 1000, 0, n - 1))])
    x = f_ev[f"ev/{side}/x"][lo:hi]; y = f_ev[f"ev/{side}/y"][lo:hi]; p = f_ev[f"ev/{side}/p"][lo:hi]
    W, H = (f_ev[f"ev/{side}/resolution"][:] if f"ev/{side}/resolution" in f_ev else (640, 480))
    surf = np.full((int(H), int(W), 3), 25, np.uint8)
    if len(x):
        surf[y[p == 0], x[p == 0]] = (255, 99, 71)          # OFF -> tomato
        surf[y[p == 1], x[p == 1]] = (0, 191, 255)          # ON  -> deepskyblue
    return surf, len(x)
ev_l, n_l = event_image("left",  t0)
ev_r, n_r = event_image("right", t0)

# LiDAR scan nearest t0 -> bird's-eye view, colored by height (z)
kl = near(f["ouster/t"][:], t0)
P = f["ouster/range_pcl"][kl].astype(np.float32) / 1000.0   # XYZ in metres (lidar frame)
P = P[np.linalg.norm(P, axis=1) > 0]

# IMU accel: +/-0.5 s window around t0 (time series; mark the instant)
it = f["vectornav/t"][:]
w  = (it > t0 - 0.5) & (it < t0 + 0.5)
tw, acc = it[w], f["vectornav/accel"][:][w]

# plot: every modality at the same instant
fig, axs = plt.subplots(2, 4, figsize=(22, 9))
def show(ax, im, title): ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis("off")
show(axs[0, 0], rgb_l, f"RGB left   t={f['img/left/t'][il]:.3f}s")
show(axs[0, 1], rgb_r, f"RGB right  t={f['img/right/t'][ir_]:.3f}s")
show(axs[0, 2], ev_l,  f"Event left  +/-20 ms  ({n_l:,} ev)")
show(axs[0, 3], ev_r,  f"Event right +/-20 ms  ({n_r:,} ev)")
show(axs[1, 0], ir_im, f"Infrared   t={f['infrared/t'][ii]:.3f}s")

ax = axs[1, 1]                                               # LiDAR bird's-eye view
ax.scatter(P[:, 0], P[:, 1], c=P[:, 2], cmap="turbo", s=0.2, vmin=-3, vmax=5)
ax.plot(0, 0, "r^", ms=8); ax.set_aspect("equal")
ax.set_xlim(-40, 40); ax.set_ylim(-40, 40); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"LiDAR BEV   t={f['ouster/t'][kl]:.3f}s")

ax = axs[1, 2]                                               # IMU accel window
for j, c in enumerate("xyz"):
    ax.plot(tw, acc[:, j], label=c)
ax.axvline(t0, color="k", ls="--", lw=1)
ax.set_title("IMU accel"); ax.set_xlabel("t (s)"); ax.set_ylabel("m/s$^2$")
ax.legend(fontsize=7, loc="upper right"); ax.grid(alpha=0.3)

ax = axs[1, 3]                                               # GPS track, position at t0
gd, gt = f["gps/data"][:], f["gps/t"][:]
gv = gd[:, 6] >= 0                                           # valid fixes
if gv.sum() > 1:
    lat, lon = gd[gv, 0], gd[gv, 1]
    gx, gy = pyproj.Transformer.from_crs(4326, 32618, always_xy=True).transform(lon, lat)

    ax.plot(gx, gy, "-", color="0.75", lw=1, zorder=2)
    ax.scatter(gx, gy, c=gt[gv], cmap="viridis", s=4, zorder=3)
    kg = near(gt[gv], t0)
    ax.plot(gx[kg], gy[kg], "*", color="red", ms=18, mec="k", zorder=4,
            label=f"t={gt[gv][kg]:.2f}s")

    ax.set_aspect("equal"); ax.legend(fontsize=8, loc="upper right")
    ax.set_title("GPS track  (star = position @ t0)")
    cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery,
                   crs=f"EPSG:{epsg}", zoom=17)
    ax.set_axis_off()
else:
    ax.text(0.5, 0.5, "no GPS fix", ha="center", va="center"); ax.set_title("GPS"); ax.axis("off")

fig.suptitle(f"Time-synchronized snapshot @ t0 = {t0:.3f}s   (each panel = nearest sample)", fontsize=15)
plt.tight_layout(); plt.show()

## Ground truth

All three ground-truth modalities are rendered in the **rectified left-RGB image** (OpenCV `stereoRectify`, `R1` applied), so they align pixel-for-pixel with the rectified left camera. Each is indexed per LiDAR scan, and `rgb_left_rect_depth.h5` carries the alignment arrays you need:
- `K_rect` (3x3) and `R3`: rectified intrinsics and rectification rotation.
- `left_img_indices[k]`: the `img_left.mp4` frame that scan `k` corresponds to.
- `poses[k]` (4x4): the LiDAR pose in the world frame at scan `k` (used to derive flow).

**Depth** (`rgb_left_rect_depth.h5 : depth_cm`, uint16 cm, `0` = invalid). Sparse **metric** depth from LiDAR. Built by accumulating **61 LiDAR scans (about 6 s)**, removing dynamic objects (YOLO26-medium on the nearest RGB frame), keeping the **minimum** depth per pixel, then projecting into the rectified left camera. Divide by 100 for meters.

**Semantic segmentation** (`rgb_left_rect_semantic.h5 : semantic`, uint8 train-IDs). 19-class **Cityscapes** pseudo-labels from [**EoMT**](https://github.com/tue-mps/eomt); label `>= 19` (or `255`) is ignore/void. Available on the **303 daytime sequences only** (`has_seg = true`); night and degraded sequences have no labels.

**Optical flow** (derived, not stored). We provide the **ego-motion-induced** flow: back-project each depth pixel to 3D, transform it by the relative camera pose between scan `k` and `k+1` (from `poses`), and reproject. The pixel displacement is the flow. This is exact for the **static scene**; independently moving objects deviate from it (one of the reason we mask these moving objects in the depth). The cell visualizes the flow with a color wheel (hue = direction, brightness = magnitude).

> **Frames recap:** depth, segmentation, and flow are all in the rectified-left image; the raw `img_*.mp4` pixels are **un-rectified**, so rectify them with `K_rect` and `R3` before overlaying. LiDAR XYZ is in the LiDAR frame; bring it into the rectified-left frame with `ouster/imgl_T_ouster` and `R3`.

In [ ]:
def turbo(z, lo=2, hi=60):
    n = (np.clip((np.clip(z, lo, hi) - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    return cv2.applyColorMap(n.reshape(-1, 1), cv2.COLORMAP_TURBO).reshape(-1, 3)[:, ::-1]

def flow_to_rgb(du, dv, max_mag):
    """Map flow vectors -> RGB: hue = direction, value = magnitude (Middlebury-style)."""
    du, dv = np.asarray(du, np.float32), np.asarray(dv, np.float32)
    ang = (np.arctan2(dv, du) + np.pi) / (2 * np.pi)        # 0..1
    mag = np.clip(np.hypot(du, dv) / max_mag, 0, 1)
    hsv = np.stack([ang * 179, np.full_like(ang, 255), mag * 255], -1).astype(np.uint8)
    return cv2.cvtColor(hsv.reshape(1, -1, 3), cv2.COLOR_HSV2RGB).reshape(*ang.shape, 3)

def color_wheel(size=201):
    g = np.linspace(-1, 1, size); xx, yy = np.meshgrid(g, g)  # yy down-positive (matches imshow)
    rgb = flow_to_rgb(xx, yy, max_mag=1.0)
    rgb[np.hypot(xx, yy) > 1] = 255                          # white outside the unit disk
    return rgb
# Cityscapes-19 palette (RGB)
CS = np.array([(128,64,128),(244,35,232),(70,70,70),(102,102,156),(190,153,153),
               (153,153,153),(250,170,30),(220,220,0),(107,142,35),(152,251,152),
               (70,130,180),(220,20,60),(255,0,0),(0,0,142),(0,0,70),
               (0,60,100),(0,80,100),(0,0,230),(119,11,32)], np.uint8)

rgbd = h5py.File(paths["rgb_left_rect_depth.h5"], "r")
rgbs = h5py.File(paths["rgb_left_rect_semantic.h5"], "r")
idx=100
img_idx=rgbs['left_img_indices'][idx]

depth = rgbd["depth_cm"][idx].astype(np.float32)
m = depth > 0
seg = rgbs["semantic"][idx]

# Undistort and Rectify left image
D_img_left = f["img/left/dist_coeffs"][:]
intr_img_left = f["img/left/intrinsics"][:]
res = f["img/left/resolution"][:]

imgr_T_imgl = np.linalg.inv(f['calib']['imgl_T_imgr'][:])
(rectl_R_rawl, rectr_R_rawr,
 P_rect_left, P_rect_right) = cv2.stereoRectify(
    cameraMatrix1=intr_img_left,  cameraMatrix2=intr_img_right,
    distCoeffs1=D_img_left,       distCoeffs2=D_img_right,
    imageSize=res,
    R=imgr_T_imgl[:3, :3], T=imgr_T_imgl[:3, -1],
    flags=cv2.CALIB_ZERO_DISPARITY, alpha=0)[:4]

mapx_left, mapy_left = cv2.initUndistortRectifyMap(
    intr_img_left,  D_img_left,  rectl_R_rawl, P_rect_left,  res, cv2.CV_32FC1)

# --- rectify a frame ---
img_raw = vl[img_idx].permute(1, 2, 0).cpu().numpy()
rectl = cv2.remap(img_raw, mapx_left, mapy_left, cv2.INTER_LINEAR)

plt.figure(figsize=(18, 6))
plt.subplot(1,3,1)
plt.title("Semantic Overlay")
seg_rgb = np.zeros((*seg.shape, 3), np.uint8)
v = seg < 19
seg_rgb[v] = CS[seg[v]]
blend = cv2.addWeighted(rectl, 0.5, seg_rgb, 0.5, 0)
blend[~v] = rectl[~v]
plt.imshow(blend); plt.axis('off')

plt.subplot(1,3,2)
plt.title("Depth GT Overlay")
B = rectl.copy(); ys, xs = np.where(m); B[ys, xs] = turbo(depth[m] / 100.0)
plt.imshow(B); plt.axis('off')

plt.subplot(1,3,3)
plt.title("Egomotion Flow")
K = rgbd["K_rect"][:]
RC = np.eye(4); RC[:3, :3] = rgbd["R3"][:]
rect_T_ous = RC @ f["ouster/imgl_T_ouster"][:]

# world<-rect-cam at idx and idx+1
wTc0 = rgbd["poses"][idx]     @ np.linalg.inv(rect_T_ous)
wTc1 = rgbd["poses"][idx + 1] @ np.linalg.inv(rect_T_ous)
T = np.linalg.inv(wTc1) @ wTc0
R, t = T[:3, :3], T[:3, 3]

# Back-project pixels to 3D using K_inv
ys, xs = np.where(m)
Z = depth[ys, xs] / 100.0
# Standard Backprojection: P = Z * inv(K) * [u, v, 1]^T
pts_2d = np.stack([xs, ys, np.ones_like(xs)], axis=0)
P = Z * (np.linalg.inv(K) @ pts_2d)

# Forward-project to new camera pose
P2 = R @ P + t[:, None]
ok = P2[2] > 1e-3
# Standard Projection: p2 = K * P2 / Z2
pts_2d_next = (K @ P2[:, ok])
pts_2d_next/=pts_2d_next[2]

du, dv = pts_2d_next[0] - xs[ok], pts_2d_next[1] - ys[ok]   # already the ok-subset
mag = np.hypot(du, dv)
xo, yo = xs[ok], ys[ok]                                      # matching pixel positions

s = slice(None, None, max(1, len(mag) // 2500))             # subsample to ~2.5k arrows
max_mag = np.percentile(mag, 95) + 1e-6
cols = flow_to_rgb(du[s], dv[s], max_mag) / 255.0          # per-arrow RGB

plt.imshow((rectl * 0.7).astype(np.uint8))
plt.quiver(xo[s], yo[s], du[s], dv[s], color=cols,
            angles="xy", scale_units="xy", scale=1, width=0.002)

axw = plt.gca().inset_axes([0.73, 0.73, 0.25, 0.25])
axw.imshow(color_wheel()); axw.axis("off")
axw.set_title("dir→hue, |flow|→val", fontsize=7)
plt.tight_layout(); plt.show()

## Further reading and citations

Methods, datasheets, and formats used in this dataset:
- [RKO-LIO](https://docs.ros.org/en/jazzy/p/rko_lio/): LiDAR-inertial odometry (the `ouster/odom` poses and the per-scan deskew).
- [EoMT](https://github.com/tue-mps/eomt): the model behind the Cityscapes pseudo-labels.
- [Cityscapes class definitions](https://www.cityscapes-dataset.com/dataset-overview/#class-definitions): the 19-class taxonomy used by the segmentation labels.
- [Trucco, E., & Verri, A. (1998). Introductory techniques for 3-D computer vision.](https://www.semanticscholar.org/paper/Introductory-techniques-for-3-D-computer-vision-Trucco-Verri/3e95b708f5f138252f84d8749a5b89cfb5c15dca): review optical flow
- [Kalibr](https://github.com/ethz-asl/kalibr): the camera and camera-IMU calibration format released alongside the data.
- [opendbc](https://github.com/commaai/opendbc): the DBC files used to decode the CAN-bus signals.
- [VectorNAV VN-100 Manual](https://www.navtechgps.com/wp-content/uploads/assets/1/7/VN100-T_UserManual-UM001.pdf)
- [Ouster OS1-64 Datasheet](https://data.ouster.io/downloads/datasheets/datasheet-rev7p1-v3p2-os0.pdf)
- [LiDAR Blooming](https://www.teachkidsrobotics.com/blog/what-are-lidar-ghosts-and-blooming/): you will see LiDAR blooming in some of our depth ground truth.
- [uBlox Zed F9P](https://cdn.sparkfun.com/assets/f/8/d/6/d/ZED-F9P-02B_DataSheet_UBX-21023276.pdf)
- [Principles of GNSS, inertial, and multisensor integrated navigation systems](https://www.amazon.com/Principles-Multi-Sensor-Integrated-Navigation-Applications/dp/1580532551): review GPS frame of referneces
- [SilkyEV Event Cameras](https://www.centuryarks.com/images/product/sensor/silkyevcam/SilkyEvCam-USB_Spec_Rev102.pdf)
- [Review of Event Cameras](https://arxiv.org/abs/1904.08405)
- [Flir RGB Cameras](https://www.teledynevisionsolutions.com/products/blackfly-s-usb3/?model=BFS-U3-28S5C-C&vertical=machine+vision&segment=iis&aPage=1)
- [Flir A35 IR Camera](https://www.flir.com/products/a35/)

License: MIT. Questions or issues: open a discussion on the [dataset page](https://huggingface.co/datasets/anthonytec2/OctoSense).

If you use OctoSense, please cite:
```bibtex
@misc{bisulco2026octosense,
  title        = {{OctoSense}: Self-Supervised Learning for Multimodal Robot Perception},
  author       = {Bisulco, Anthony and Wang, Jeremy and Daniilidis, Kostas and Balestriero, Randall and Chaudhari, Pratik},
  year         = {2026},
  howpublished = {Preprint},
}
```